# 01 · Load and profile all five data sources

Week 1 checkpoint. Loads every file, then profiles each so the quality problems are visible before any cleaning.

In [1]:
import os, sqlite3
import pandas as pd

DATA_DIR = next(d for d in ('data', '../data', '../../data') if os.path.isdir(d))

def profile(df, name):
    print(f'=== {name} ===')
    print(f'{len(df):,} rows x {len(df.columns)} columns')
    summary = pd.DataFrame({'dtype': df.dtypes.astype(str), 'nulls': df.isnull().sum(),
                            'null_pct': (100 * df.isnull().mean()).round(1), 'distinct': df.nunique()})
    print(summary)
    print('exact duplicate rows:', df.duplicated().sum(), '\n')
    return summary

In [2]:
tx = pd.read_csv(os.path.join(DATA_DIR, 'walmart_transactions.csv'))
stores = pd.read_csv(os.path.join(DATA_DIR, 'store_locations.csv'))
catalog = pd.read_csv(os.path.join(DATA_DIR, 'product_catalog (1).csv'))
dcs = pd.read_csv(os.path.join(DATA_DIR, 'distribution_centers.csv'))

con = sqlite3.connect(os.path.join(DATA_DIR, 'inventory_and_sales.db'))
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con))
daily_sales = pd.read_sql('SELECT * FROM daily_sales_summary', con)
inventory = pd.read_sql('SELECT * FROM inventory_levels', con)
con.close()

for df, name in [(tx, 'walmart_transactions'), (stores, 'store_locations'), (catalog, 'product_catalog'),
                 (dcs, 'distribution_centers'), (daily_sales, 'daily_sales_summary'), (inventory, 'inventory_levels')]:
    profile(df, name)

                  name
0  daily_sales_summary
1     inventory_levels
=== walmart_transactions ===
25,562 rows x 8 columns
                dtype  nulls  null_pct  distinct
transaction_id    str      0       0.0     25532
store_id          str      0       0.0        60
timestamp         str    886       3.5     23952
sku               str      0       0.0        45
product_name      str      0       0.0        63
category          str      0       0.0        37
quantity        int64      0       0.0        11
unit_price        str      0       0.0      3668
exact duplicate rows: 30 

=== store_locations ===
14 rows x 8 columns
              dtype  nulls  null_pct  distinct
store_id        str      0       0.0        14
store_name      str      0       0.0        12
city            str      0       0.0        12
state           str      0       0.0         3
zip_code      int64      0       0.0        12
latitude    float64      1       7.1        11
longitude   float64      1       7.1 

## What the profile shows

Transactions:
- `store_id` has 60 distinct values for 12 real stores (case, `_`, missing hyphen, leading spaces).
- `category` has 37 spellings for about 10 categories.
- `product_name` has 63 variants and `sku` has 45 values for roughly 20 products.
- `timestamp` has 886 missing values and mixed formats.
- `unit_price` is text (some `$10.24`, some padded with spaces).
- 30 exact duplicate rows; negative `quantity` values.

In [3]:
print('quantity value counts:'); print(tx['quantity'].value_counts().sort_index().to_dict())
print('\nunit_price examples with $ or spaces:'); print(tx.loc[tx['unit_price'].str.contains(r'\$|^\s|\s$', regex=True), 'unit_price'].head(8).tolist())
print('\nstore_locations rows:', len(stores), '| distinct store_id spellings in tx:', tx['store_id'].nunique())
print('\nstore_locations problems:'); print(stores[stores['latitude'].isna() | stores['store_id'].duplicated(keep=False) | ~stores['store_id'].str.match(r'^FL-\d{3}$')])

quantity value counts:
{-5: 4, -4: 4, -3: 15, -2: 53, -1: 54, 1: 10268, 2: 9290, 3: 3745, 4: 1343, 5: 528, 6: 258}

unit_price examples with $ or spaces:
['$4.76', '$8.75', ' 19.32 ', '$3.31', '$7.82', ' 3.45 ', '$13.02', '$19.16']

store_locations rows: 14 | distinct store_id spellings in tx: 60

store_locations problems:
   store_id                             store_name         city    state  \
8    FL-619        Walmart Supercenter Tallahassee  Tallahassee       FL   
12    FL239            Walmart Supercenter Orlando      Orlando       FL   
13   FL_287  Walmart Neighborhood Market Kissimmee    Kissimmee  Florida   

    zip_code  latitude  longitude      region  
8      32301       NaN        NaN   Panhandle  
12     32801   28.5383   -81.3792     Central  
13     34741   28.2920   -81.4076     Central  
